# External validation of DEME's representation (MFRC)

Three tests that the multi-dimensional representation is real, not an artifact, using the
**Moral Foundations Reddit Corpus** (human-annotated). For 280 stratified comments we scored
DEME's dimensions via the NRP LLM panel on three *instruments* (10-module, canonical 9-axis,
7-axis harm space) and two *model families* (qwen3, glm-5). Data: `mfrc_multi.jsonl` (derived
scores + human label fractions, no raw text; source HF `USC-MOLA-Lab/MFRC`).

1. **Alignment** - DEME dimensions recover human moral foundations (pre-registered).
2. **Multi-model robustness** - the alignment replicates across two independent models.
3. **Cross-instrument core** - different instruments agree on the shared core (Appendix A).


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

D = [json.loads(l) for l in open('mfrc_multi.jsonl', encoding='utf-8')]
FOUND = ['Care', 'Equality', 'Proportionality', 'Authority', 'Loyalty', 'Purity']
DIMS10 = ['physical_harm', 'rights_respect', 'fairness_equity', 'autonomy_consent',
          'legitimacy_trust', 'epistemic_quality', 'care_protection', 'vow_fidelity',
          'third_party_externality', 'repair_residue']
H = np.array([[r['frac'][f] for f in FOUND] for r in D])
def col(inst, dim): return np.array([r[inst][dim] for r in D])
rng = np.random.default_rng(0)
def boot_ci(x, y, n=1000):
    rs = [spearmanr(x[ix], y[ix])[0] for ix in (rng.integers(0, len(x), len(x)) for _ in range(n))]
    return np.percentile(rs, [2.5, 97.5])
align = [('care_protection', 'Care'), ('fairness_equity', 'Equality'),
         ('legitimacy_trust', 'Authority'), ('vow_fidelity', 'Loyalty')]
print(f'{len(D)} comments; instruments m10_qwen / m9_qwen / m7_qwen / m10_glm')

## 1+2. Pre-registered alignment, with bootstrap CIs, replicated across two models


In [ ]:
xs = np.arange(len(align)); fig, ax = plt.subplots(figsize=(8.5, 3.8))
for inst, color, off in [('m10_qwen', 'tab:green', -0.2), ('m10_glm', 'tab:olive', 0.2)]:
    rhos, lo, hi = [], [], []
    for dim, f in align:
        h = H[:, FOUND.index(f)]; r = spearmanr(col(inst, dim), h)[0]; ci = boot_ci(col(inst, dim), h)
        rhos.append(r); lo.append(r - ci[0]); hi.append(ci[1] - r)
    ax.bar(xs + off, rhos, 0.4, yerr=[lo, hi], capsize=3, color=color, label=inst.split('_')[1])
ax.set_xticks(xs); ax.set_xticklabels([d + chr(10) + '~ ' + f for d, f in align], fontsize=8)
ax.set_ylabel('Spearman rho (95% CI)')
ax.set_title('DEME dimensions recover human foundations - replicated across qwen3 and glm-5')
ax.legend(); plt.tight_layout(); plt.show()
inter = [spearmanr(col('m10_qwen', d), col('m10_glm', d))[0] for d in DIMS10]
print(f'inter-model (qwen3 vs glm-5) per-dimension agreement: mean rho={np.mean(inter):.2f}, min={min(inter):.2f}')

## 3. Cross-instrument core - do different instruments share the same core? (Appendix A)


In [ ]:
CORE = {'Harm': ('physical_harm', 'physical_harm'), 'Rights': ('rights_respect', 'rights_respect'),
        'Fairness': ('fairness_equity', 'fairness_equity'), 'Autonomy': ('autonomy_consent', 'autonomy_respect'),
        'Legitimacy': ('legitimacy_trust', 'legitimacy_trust'), 'Epistemic': ('epistemic_quality', 'epistemic_quality'),
        'Care': ('care_protection', 'virtue_care')}
axesc = list(CORE)
agr = [spearmanr(col('m10_qwen', CORE[a][0]), col('m9_qwen', CORE[a][1]))[0] for a in axesc]
harm7 = (col('m7_qwen', 'physical') + col('m7_qwen', 'emotional') + col('m7_qwen', 'financial')) / 3
sub = [('Harm.7ax', harm7, col('m10_qwen', 'physical_harm')),
       ('Autonomy.7ax', col('m7_qwen', 'autonomy'), col('m10_qwen', 'autonomy_consent')),
       ('Legitimacy.7ax', col('m7_qwen', 'trust'), col('m10_qwen', 'legitimacy_trust'))]
sub_r = [spearmanr(a, b)[0] for _, a, b in sub]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(axesc, agr, color='tab:blue', label='9-axis vs 10-module (full core)')
ax.bar([s[0] for s in sub], sub_r, color='tab:cyan', label='7-axis harm vs 10-module (subcore)')
ax.axhline(np.mean(agr), color='k', ls='--', lw=0.8, label=f'mean 9v10 = {np.mean(agr):.2f}')
ax.set_ylabel('cross-instrument Spearman rho')
ax.set_title('Different instruments agree on the shared moral core')
plt.xticks(rotation=30, ha='right'); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f'mean 9-axis vs 10-module core agreement: {np.mean(agr):.3f}')

## The full diagonal: DEME dimensions x human foundations


In [ ]:
M = np.array([[spearmanr(col('m10_qwen', d), H[:, j])[0] for j in range(len(FOUND))] for d in DIMS10])
fig, ax = plt.subplots(figsize=(7, 7.5)); im = ax.imshow(M, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
ax.set_xticks(range(len(FOUND))); ax.set_xticklabels(FOUND, rotation=45, ha='right')
ax.set_yticks(range(len(DIMS10))); ax.set_yticklabels(DIMS10)
for i in range(len(DIMS10)):
    for j in range(len(FOUND)):
        ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(M[i, j]) > 0.3 else 'black')
for dim, f in align:
    ax.add_patch(plt.Rectangle((FOUND.index(f) - 0.5, DIMS10.index(dim) - 0.5), 1, 1, fill=False, edgecolor='lime', lw=2.5))
fig.colorbar(im, ax=ax, shrink=0.6, label='rho')
ax.set_title('DEME dims x human foundations (green = pre-registered)')
plt.tight_layout(); plt.show()

## Conclusion

DEME's representation passes three independent tests against human-annotated MFRC data: the
dimensions **align** with their intended moral foundations (each the argmax, rho ~ 0.43-0.53,
p<1e-15); the alignment **replicates** across two model families with overlapping 95% CIs
(ruling out a single-LLM artifact); and distinct **instruments agree on the shared core**
(9-axis vs 10-module mean rho=0.88; the differently-based 7-axis harm space recovers the
subcore) - the empirical support for Appendix A that the paper's Limitations flagged as
outstanding. Reproducible from `mfrc_multi.jsonl` + `score_mfrc_multi.py` + MFRC.
